# NB57: Hot/Cold Architecture

Routing Hot data to ES and Cold data to MinIO.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **Elasticsearch**: Search and analytics engine.
- **MinIO**: S3-compatible object storage.

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Elasticsearch
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.10.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.10.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.10.2
!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" ./elasticsearch-7.10.2/bin/elasticsearch -d > es.log 2>&1 &
# Start MinIO
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!mkdir -p /content/minio_data
!MINIO_ROOT_USER=minioadmin MINIO_ROOT_PASSWORD=minioadmin ./minio server /content/minio_data --console-address ":9001" &> minio.log &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9000) # MinIO
time.sleep(5) # Extra buffer for MinIO
wait_for_port(9200) # Elasticsearch


## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Sends data tagged as 'hot' or 'cold'.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Hot/Cold Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 items...")
for i in range(100):
    data = {'type': random.choice(['hot', 'cold']), 'val': i}
    producer.send('input-topic', json.dumps(data).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Spark Hot/Cold Routing

Routes 'hot' data to Elasticsearch and 'cold' data to MinIO.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from elasticsearch import Elasticsearch
from minio import Minio
import json, io

spark = SparkSession.builder.appName("Router").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    es = Elasticsearch(['http://localhost:9200'])
    m = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
    if not m.bucket_exists("cold"): m.make_bucket("cold")
    for row in rows:
        d = json.loads(row.value)
        if d['type'] == 'hot':
            es.index(index='hot_data', body=d)
        else:
            m.put_object("cold", f"obj_{d['val']}", io.BytesIO(row.value), len(row.value))
    print(f"Batch {epoch_id} routed.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check both destinations.

In [ ]:
from elasticsearch import Elasticsearch
from minio import Minio

# Check ES
es = Elasticsearch(['http://localhost:9200'])
time.sleep(2)
res = es.count(index="hot_data")
print(f"Hot Data (ES): {res['count']} docs")

# Check MinIO
m = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
objs = list(m.list_objects("cold"))
print(f"Cold Data (MinIO): {len(objs)} files")